# NIFTY 50 Correction-Risk ML Project

This notebook trains and compares **Logistic Regression**, **Random Forest**, and a simple **India VIX threshold rule** using the supplied dataset.

## Method
- Sort rows by date and build causal features from current/past observations only.
- Drop rows with missing targets.
- Split chronologically into older training, validation, and newest test partitions (approximately 70/15/15), with a 20-row purge at each boundary because the target looks 20 rows ahead.
- Fit median imputation (and scaling for Logistic Regression) on training data only.
- Choose each classifier's probability threshold and the VIX-rule threshold using validation data only.
- Evaluate once on the untouched test period.

**Important limitation:** The target labels are accepted as supplied. This notebook measures how well models reproduce those labels; it does not validate that labels represent real-world correction risk.


In [ ]:
"""
NIFTY 50 correction-risk classification project
Run:
    pip install pandas numpy scikit-learn matplotlib jupyter
    python nifty_correction_model.py --input "final 2(1).csv" --output_dir outputs

Important:
- The target is taken from the provided file, not reconstructed here.
- Predictor features are rebuilt using current/past rows only.
- The target's real-world correctness is not assumed.
- Chronological splits include 20-row purges between partitions because the target looks 20 rows ahead.
"""
import argparse
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score,
    balanced_accuracy_score, average_precision_score, roc_auc_score,
    confusion_matrix, precision_recall_curve
)

RANDOM_STATE = 42
HORIZON = 20
TARGET = "correction_within_20d"

def build_features(raw):
    """Sort chronologically and create only causal features from raw columns."""
    data = raw.copy()
    data["date"] = pd.to_datetime(data["date"], errors="coerce")
    data = data.dropna(subset=["date"]).sort_values("date", kind="stable").reset_index(drop=True)

    # Ensure numerical inputs are numeric. Existing helper columns are intentionally not used.
    numeric_source_cols = [
        "nifty50_open", "nifty50_high", "nifty50_low", "nifty50_close", "nifty50_pe",
        "india_vix_close", "usd_inr_open", "usd_inr_high", "usd_inr_low",
        "usd_inr_close", "usd_inr_reference_rate"
    ]
    for col in numeric_source_cols:
        if col in data.columns:
            data[col] = pd.to_numeric(data[col], errors="coerce")

    # Causal returns: current close relative to a close in the past.
    data["nifty_return_1d_recalc"] = data["nifty50_close"].pct_change(1, fill_method=None)
    data["nifty_return_5d_recalc"] = data["nifty50_close"].pct_change(5, fill_method=None)
    data["nifty_return_20d_recalc"] = data["nifty50_close"].pct_change(20, fill_method=None)
    data["nifty_return_60d_recalc"] = data["nifty50_close"].pct_change(60, fill_method=None)
    data["nifty_volatility_20d_recalc"] = data["nifty_return_1d_recalc"].rolling(20, min_periods=20).std()

    data["vix_return_1d_recalc"] = data["india_vix_close"].pct_change(1, fill_method=None)
    data["vix_return_5d_recalc"] = data["india_vix_close"].pct_change(5, fill_method=None)
    data["usd_inr_return_1d_recalc"] = data["usd_inr_close"].pct_change(1, fill_method=None)
    data["usd_inr_return_5d_recalc"] = data["usd_inr_close"].pct_change(5, fill_method=None)

    # Explicit allow-list. Never use target, reference_peak, drawdown, threshold,
    # days_since_peak, max_drawdown_20d, or any existing helper columns.
    features = [
        "nifty50_open", "nifty50_high", "nifty50_low", "nifty50_close", "nifty50_pe",
        "india_vix_close",
        "usd_inr_close", "usd_inr_reference_rate",
        "nifty_return_1d_recalc", "nifty_return_5d_recalc",
        "nifty_return_20d_recalc", "nifty_return_60d_recalc",
        "nifty_volatility_20d_recalc",
        "vix_return_1d_recalc", "vix_return_5d_recalc",
        "usd_inr_return_1d_recalc", "usd_inr_return_5d_recalc"
    ]
    features = [c for c in features if c in data.columns]
    data[TARGET] = pd.to_numeric(data[TARGET], errors="coerce")
    data = data[data[TARGET].isin([0, 1])].copy()
    data[TARGET] = data[TARGET].astype(int)
    return data, features

def chronological_split(data, horizon=HORIZON):
    """70/15/15 by time with a horizon-sized purge at each boundary."""
    n = len(data)
    train_end = int(n * 0.70)
    val_end = int(n * 0.85)
    train = data.iloc[:max(0, train_end - horizon)].copy()
    val = data.iloc[train_end:max(train_end, val_end - horizon)].copy()
    test = data.iloc[val_end:].copy()
    if min(len(train), len(val), len(test)) == 0:
        raise ValueError("Not enough labeled rows for chronological split.")
    return train, val, test

def choose_threshold(y_true, scores):
    """Choose probability threshold by validation F1 only."""
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    if len(thresholds) == 0:
        return 0.5
    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    best = int(np.nanargmax(f1))
    return float(thresholds[best])

def metrics_row(name, y, pred, scores, threshold):
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    try:
        pr_auc = average_precision_score(y, scores)
    except Exception:
        pr_auc = np.nan
    try:
        roc_auc = roc_auc_score(y, scores)
    except Exception:
        roc_auc = np.nan
    return {
        "model": name,
        "threshold_selected_on_validation": threshold,
        "test_n": len(y),
        "test_positive_labels": int(np.sum(y)),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "f1": f1_score(y, pred, zero_division=0),
        "accuracy": accuracy_score(y, pred),
        "balanced_accuracy": balanced_accuracy_score(y, pred),
        "pr_auc_average_precision": pr_auc,
        "roc_auc": roc_auc,
        "true_negatives": int(tn),
        "false_alarms_fp": int(fp),
        "missed_corrections_fn": int(fn),
        "true_positives": int(tp),
    }

def main(input_path, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    raw = pd.read_csv(input_path)
    if TARGET not in raw:
        raise ValueError(f"Input must contain {TARGET!r}.")
    data, features = build_features(raw)
    train, val, test = chronological_split(data)

    # Report missingness based on training features, before imputation.
    missingness = train[features].isna().mean().sort_values(ascending=False)
    missingness.rename("training_missing_fraction").to_csv(
        os.path.join(output_dir, "feature_missingness_train.csv"), header=True
    )
    # Remove a feature only if >40% missing in TRAINING; record this explicitly.
    high_missing = missingness[missingness > 0.40].index.tolist()
    features = [c for c in features if c not in high_missing]
    if high_missing:
        warnings.warn(f"Removed features with >40% training missingness: {high_missing}")

    X_train, y_train = train[features], train[TARGET]
    X_val, y_val = val[features], val[TARGET]
    X_test, y_test = test[features], test[TARGET]

    models = {
        "Logistic Regression": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE
            ))
        ]),
        "Random Forest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(
                n_estimators=500, min_samples_leaf=5, max_features="sqrt",
                class_weight="balanced_subsample", random_state=RANDOM_STATE,
                n_jobs=-1
            ))
        ])
    }

    rows, predictions, fitted = [], {}, {}
    for name, pipe in models.items():
        pipe.fit(X_train, y_train)  # imputer/scaler fit on training only
        val_scores = pipe.predict_proba(X_val)[:, 1]
        threshold = choose_threshold(y_val, val_scores)  # validation only
        test_scores = pipe.predict_proba(X_test)[:, 1]
        test_pred = (test_scores >= threshold).astype(int)
        rows.append(metrics_row(name, y_test.to_numpy(), test_pred, test_scores, threshold))
        predictions[name] = (test_scores, test_pred, threshold)
        fitted[name] = pipe

    # Simple rule baseline: VIX >= threshold. Select threshold on validation only.
    if "india_vix_close" in features:
        vix_candidates = np.unique(np.nanquantile(
            val["india_vix_close"].dropna(), np.linspace(0.05, 0.95, 91)
        ))
        best_rule = None
        for th in vix_candidates:
            # Use the training median only as fallback for a missing VIX observation.
            fallback = train["india_vix_close"].median()
            val_vix = val["india_vix_close"].fillna(fallback).to_numpy()
            val_pred = (val_vix >= th).astype(int)
            f = f1_score(y_val, val_pred, zero_division=0)
            if best_rule is None or f > best_rule[0]:
                best_rule = (f, float(th))
        vix_threshold = best_rule[1]
        fallback = train["india_vix_close"].median()
        test_vix = test["india_vix_close"].fillna(fallback).to_numpy()
        rule_pred = (test_vix >= vix_threshold).astype(int)
        # Binary rule's score is its 0/1 output, so PR-AUC is reported but less informative.
        rows.append(metrics_row(
            "India VIX threshold rule", y_test.to_numpy(), rule_pred,
            rule_pred.astype(float), vix_threshold
        ))
        predictions["India VIX threshold rule"] = (rule_pred.astype(float), rule_pred, vix_threshold)

    results = pd.DataFrame(rows)
    results.to_csv(os.path.join(output_dir, "test_model_comparison.csv"), index=False)

    # Confusion matrices saved to CSV.
    for name, (_, pred, _) in predictions.items():
        cm = confusion_matrix(y_test, pred, labels=[0, 1])
        safe = name.lower().replace(" ", "_")
        pd.DataFrame(cm, index=["actual_no_correction", "actual_correction"],
                     columns=["predicted_no_correction", "predicted_correction"]).to_csv(
            os.path.join(output_dir, f"confusion_matrix_{safe}.csv")
        )

    # Precision-recall curve on the untouched test period.
    plt.figure(figsize=(8, 6))
    for name, (scores, _, _) in predictions.items():
        if name == "India VIX threshold rule":
            # Step curve from the binary rule has only two score levels; still show the baseline.
            precision, recall, _ = precision_recall_curve(y_test, scores)
        else:
            precision, recall, _ = precision_recall_curve(y_test, scores)
        plt.plot(recall, precision, label=name)
    prevalence = float(y_test.mean())
    plt.axhline(prevalence, linestyle="--", label=f"No-skill prevalence ({prevalence:.2f})")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision–Recall Curves (Untouched Test Period)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "precision_recall_curves.png"), dpi=160)
    plt.close()

    # Feature interpretation.
    rf = fitted["Random Forest"].named_steps["model"]
    rf_imp = pd.DataFrame({
        "feature": features, "importance": rf.feature_importances_
    }).sort_values("importance", ascending=False)
    rf_imp.to_csv(os.path.join(output_dir, "random_forest_feature_importance.csv"), index=False)

    lr = fitted["Logistic Regression"].named_steps["model"]
    lr_coef = pd.DataFrame({
        "feature": features,
        "coefficient": lr.coef_[0],
        "odds_ratio_per_scaled_unit": np.exp(np.clip(lr.coef_[0], -20, 20))
    }).sort_values("coefficient", key=lambda s: s.abs(), ascending=False)
    lr_coef.to_csv(os.path.join(output_dir, "logistic_regression_coefficients.csv"), index=False)

    # Save split/date/missingness/target summaries.
    summary = {
        "input_file": os.path.basename(input_path),
        "raw_rows": int(len(raw)),
        "raw_date_min": str(pd.to_datetime(raw["date"], errors="coerce").min().date()),
        "raw_date_max": str(pd.to_datetime(raw["date"], errors="coerce").max().date()),
        "rows_with_labeled_target": int(len(data)),
        "target_missing_rows_dropped": int(raw[TARGET].isna().sum()),
        "target_class_counts_labeled": {str(k): int(v) for k, v in data[TARGET].value_counts().sort_index().items()},
        "features_used": features,
        "features_removed_for_gt_40pct_training_missingness": high_missing,
        "train_rows": int(len(train)),
        "validation_rows": int(len(val)),
        "test_rows": int(len(test)),
        "train_date_range": [str(train["date"].min().date()), str(train["date"].max().date())],
        "validation_date_range": [str(val["date"].min().date()), str(val["date"].max().date())],
        "test_date_range": [str(test["date"].min().date()), str(test["date"].max().date())],
        "purge_rows_each_boundary": HORIZON,
        "note": "Performance is conditional on supplied target labels and source data; not evidence of reliable real-world forecasting."
    }
    with open(os.path.join(output_dir, "run_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    print("\n=== TEST PERFORMANCE (thresholds chosen on validation only) ===")
    print(results.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
    print("\n=== SPLIT SUMMARY ===")
    print(json.dumps(summary, indent=2))
    print("\nSaved outputs to:", os.path.abspath(output_dir))


In [ ]:
# Set this to the path of the CSV in your environment.
INPUT_FILE = "final 2(1).csv"
OUTPUT_DIR = "nifty_ml_outputs"
main(INPUT_FILE, OUTPUT_DIR)


## Inspect the results

In [ ]:
from IPython.display import display, Image
results = pd.read_csv(os.path.join(OUTPUT_DIR, "test_model_comparison.csv"))
display(results.round(3))
display(pd.read_csv(os.path.join(OUTPUT_DIR, "confusion_matrix_logistic_regression.csv"), index_col=0))
display(pd.read_csv(os.path.join(OUTPUT_DIR, "confusion_matrix_random_forest.csv"), index_col=0))
display(pd.read_csv(os.path.join(OUTPUT_DIR, "confusion_matrix_india_vix_threshold_rule.csv"), index_col=0))


In [ ]:
display(Image(filename=os.path.join(OUTPUT_DIR, "precision_recall_curves.png")))
display(pd.read_csv(os.path.join(OUTPUT_DIR, "random_forest_feature_importance.csv")).head(12))
display(pd.read_csv(os.path.join(OUTPUT_DIR, "logistic_regression_coefficients.csv")).head(12))


## Interpretation checklist
1. Compare missed corrections (false negatives) with false alarms (false positives), not accuracy alone.
2. Check whether Logistic Regression or Random Forest is preferable for the project's intended warning use.
3. Compare each ML model with the VIX rule.
4. Do not claim reliable real-world prediction until date alignment and target-label definitions are validated.
